In [32]:
#imports
import pandas as pd
from pathlib import Path
import numpy as np
print("Import completed")

Import completed


Create the final dataframe that contains the filtered data and requirements of the SRS that will go here

Inputs (already on main, no need to rerun anything)

data/addressable_clauses.csv: the 62 NASA clauses an SRS can be checked against (keyed by chunk_id)

data/Starhawk Mission Computer SRS.md: the 54 requirements, SFMC-REQ-001 to 054

Suggested notebook: notebooks/03_build_requirement_pairs.ipynb

-----------------------------------------------------------------------------------------------------------------------------------------------------------

1. Requirements table → data/srs_requirements.csv
One row per requirement: req_id, title, text, section (e.g. "3.1 Navigation")

Each requirement looks like **SFMC-REQ-001 — Position Display** followed by its text
REQ-013, 029 and 045 have bullet lists. Keep the list inside that requirement's text
Section 2.1 has two "shall" sentences with no ID. Leave them out (or add a note)
Check: 54 rows, IDs 001–054, no blank text

-----------------------------------------------------------------------------------------------------------------------------------------------------------

## Loading the content from the markdown file

In [33]:
#import the md file using the with open() as file method
with open("../data/Starhawk Mission Computer SRS.md", "r") as file:
    SRSRequirements = file.read()

print(SRSRequirements)

# Software Requirements Specification (SRS)
## Starhawk Space Fighter Mission Computer

**Document ID:** SH-SRS-001  
**Version:** 1.0  
**Status:** Draft for Review  

---

# 1. Introduction

## 1.1 Purpose

This Software Requirements Specification defines the software requirements for the **Starhawk Space Fighter Mission Computer (SFMC)**.

The SFMC is the primary onboard computer for the Starhawk single-seat space fighter. The software assists the pilot with spacecraft navigation, propulsion control, targeting, defensive systems, communications, and vehicle health monitoring.

This specification describes the required behavior of the SFMC software. Hardware design and detailed software implementation are outside the scope of this document.

## 1.2 System Overview

The Starhawk is a fictional spacecraft designed to operate both independently and as part of a larger fleet.

The spacecraft contains the following major subsystems:

- Mission Computer
- Navigation System
- Propulsion Sys

## Breaking down the markdown file into a form for a csv file and dataframe file 

When having to set up the Requirement table, srs_requirements.csv, there are 4 columns in that file:

req_id (ex: SFMC-REQ-001)

title (ex: Position Display)

text (ex: The SFMC shall display the spacecraft's current position to the pilot)

section (ex: 3.1 Navigation or 5. Performance Requirements)

In order to breakdown the text to get the contents needed for the table, the content that starts in 3.1 navigation and ends in 10. Requirement Traceability.

The text in that variable will be parse so that way, req_id, title, text, and section will go into their proper lists so that they can be used to make the csv file and the dataframe.

-----------------------------------------------------------------------------------------------------------------------------------------------------------

In [34]:
#grabbing the index of both 3.1 and the 10. Requirment Traceability in order to put that substring in variable TargetText
if "3.1" in SRSRequirements:
    StartingIndex = SRSRequirements.find("3.1")
else:
    print("String is not found")
print(StartingIndex)

if "10. Requirement Traceability" in SRSRequirements:
    EndingIndex = SRSRequirements.find("10. Requirement Traceability")
else:
    print("String is not found")
print(EndingIndex)

TargetText = SRSRequirements[StartingIndex:EndingIndex]
print(TargetText)

2864
10640
3.1 Navigation

**SFMC-REQ-001 â€” Position Display**

The SFMC shall display the spacecraft's current position to the pilot.

**SFMC-REQ-002 â€” Navigation Update**

The SFMC shall update the displayed spacecraft position at least once every 500 milliseconds while the spacecraft is in Manual Flight, Autopilot, or Combat mode.

**SFMC-REQ-003 â€” Destination Selection**

The SFMC shall allow the pilot to select a navigation destination.

**SFMC-REQ-004 â€” Route Calculation**

When the pilot selects a destination, the SFMC shall calculate a safe and efficient route to the destination.

**SFMC-REQ-005 â€” Autopilot**

When Autopilot mode is enabled, the SFMC shall control the spacecraft propulsion system to follow the selected route.

**SFMC-REQ-006 â€” Autopilot Arrival**

The autopilot shall deliver the spacecraft reasonably close to the selected destination.

---

## 3.2 Propulsion

**SFMC-REQ-007 â€” Engine Status**

The SFMC shall continuously monitor the status of the s

In [35]:
#for this block, there will be 4 lists, req_id_list, title_list, text_list, and section_list.
#these 4 lists and the Target will be plugged into a function called set_text
#set_text will focus on parsing the text in TargetText where it order the text to be placed in one of these lists based on whether it matches the conditions
#that makes it a req_id, a title, text, or section.

#making the lists
req_id_list = []
title_list = []
text_list = []
section_list = []

def is_numeric(string: str) -> bool:
    # Try to convert the string to a float
    # If the conversion is successful, return True
    try:
        float(string)
        return True
    # If a ValueError is thrown, it means the conversion was not successful
    # This happens when the string contains non-numeric characters
    except ValueError:
        return False
#Source: w3schools

#setting up the set_text function
#when the function spots a number, it will concat the number plus the set amount of string in that sequence until it hits **SFMC_REQ_
#Once it hits **SFMC_REQ_ then it string will be added to the section list and made empty.
#Then **SFMC_REQ_ will added to the req_id list and will skip over â€” to start title part
#for the title part, there will string that will be a string that will be concat until it hits a word with ** in it.
#that title string will be added to title list and the string will be made empty.
#For the text part, there will be a string that will keep adding words until it encounters an **SFMC_REQ_ or a number
#in order to keep track to the amount of characters there is, we will have variable skip which counts how many character the next iteration we will need
#to skip 

#revising the function since it is going over the amount that .split makes
#this new version will remove the displacment part of the orignal and instead work like this
#when the number spotted, it will added the number and the string after it until it encounters **SFMC, thats when it stops and adds the contents to section
#When **SFMC is spotted then it will add the req into Str_req_id and then Str_req_id into req_id
#When â€ is spotted, it will skip over it and instead add the text for the title via Str_title until it encounters one with ** which will be added to the
#String and then put into title
#if none of those then it will add the text into the text until encounters an **SFMC or a number and which point the text goes into text via Str_text
def set_text(TText, req_id, title, text, section):
    Text = TText.split()
    Str_section = ""
    Str_req_id = ""
    Str_title = ""
    Str_text = ""
    isApartofSection = False
    isApartofTitle = False
    isNumStart = True

    for i in Text:
        current_word = i

        if (is_numeric(current_word) == True or isApartofSection == True) and isNumStart == True:
            if "**SFMC" in current_word:
                isApartofSection = False
                isNumStart = False
                section.append(Str_section)
                Str_section = ""
                Str_req_id = current_word
                req_id.append(Str_req_id)
                Str_req_id = ""
            else:
                isApartofSection = True
                Str_section = Str_section + current_word + " "
        elif "**SFMC" in current_word:
            Str_req_id = current_word
            req_id.append(Str_req_id)
            Str_req_id = ""
        elif 'â€”' in current_word:
            isApartofTitle = True
        elif isApartofTitle == True:
            if "**" in current_word:
                Str_title = Str_title + current_word
                isApartofTitle = False
                title.append(Str_title)
                Str_title = ""
            else:
                Str_title = Str_title + current_word + " "
        elif "---" in current_word:
            continue
        elif "##" in current_word or "#" in current_word:
            isNumStart = True
        else:
            if "**SFMC" in current_word:
               text.append(Str_text)
               Str_text = ""
               Str_req_id = current_word
               req_id.append(Str_req_id)
               Str_req_id = "" 
            #elif is_numeric(current_word) == True:
                #text.append(Str_text)
                #Str_text = ""
            else:
                if "." in current_word:
                    Str_text = Str_text + current_word
                    text.append(Str_text)
                    Str_text = ""
                else:
                    Str_text = Str_text + current_word + " "

In [36]:
set_text(TargetText, req_id_list, title_list, text_list, section_list)
print(req_id_list)
print(title_list)
print(text_list)
print(section_list)

print(len(req_id_list))
print(len(title_list))
print(len(text_list))
print(len(section_list))

['**SFMC-REQ-001', '**SFMC-REQ-002', '**SFMC-REQ-003', '**SFMC-REQ-004', '**SFMC-REQ-005', '**SFMC-REQ-006', '**SFMC-REQ-007', '**SFMC-REQ-008', '**SFMC-REQ-009', '**SFMC-REQ-010', '**SFMC-REQ-011', '**SFMC-REQ-012', '**SFMC-REQ-013', '**SFMC-REQ-014', '**SFMC-REQ-015', '**SFMC-REQ-016', '**SFMC-REQ-017', '**SFMC-REQ-018', '**SFMC-REQ-019', '**SFMC-REQ-020', '**SFMC-REQ-021', '**SFMC-REQ-022', '**SFMC-REQ-023', '**SFMC-REQ-024', '**SFMC-REQ-025', '**SFMC-REQ-026', '**SFMC-REQ-027', '**SFMC-REQ-028', '**SFMC-REQ-029', '**SFMC-REQ-030', '**SFMC-REQ-031', '**SFMC-REQ-032', '**SFMC-REQ-033', '**SFMC-REQ-034', '**SFMC-REQ-035', '**SFMC-REQ-036', '**SFMC-REQ-037', '**SFMC-REQ-038', '**SFMC-REQ-039', '**SFMC-REQ-040', '**SFMC-REQ-041', '**SFMC-REQ-042', '**SFMC-REQ-043', '**SFMC-REQ-044', '**SFMC-REQ-045', '**SFMC-REQ-046', '**SFMC-REQ-047', '**SFMC-REQ-048', '**SFMC-REQ-049', '**SFMC-REQ-050', '**SFMC-REQ-051', '**SFMC-REQ-052', '**SFMC-REQ-053', '**SFMC-REQ-054']
['Position Display**', 'Nav

In [37]:
#Fixing some of the text for list req_id_list and title_list
revised_req_id_list = [x.strip('**') for x in req_id_list]
revised_title_list = [y.strip('**') for y in title_list]

print(revised_req_id_list)
print(revised_title_list)

['SFMC-REQ-001', 'SFMC-REQ-002', 'SFMC-REQ-003', 'SFMC-REQ-004', 'SFMC-REQ-005', 'SFMC-REQ-006', 'SFMC-REQ-007', 'SFMC-REQ-008', 'SFMC-REQ-009', 'SFMC-REQ-010', 'SFMC-REQ-011', 'SFMC-REQ-012', 'SFMC-REQ-013', 'SFMC-REQ-014', 'SFMC-REQ-015', 'SFMC-REQ-016', 'SFMC-REQ-017', 'SFMC-REQ-018', 'SFMC-REQ-019', 'SFMC-REQ-020', 'SFMC-REQ-021', 'SFMC-REQ-022', 'SFMC-REQ-023', 'SFMC-REQ-024', 'SFMC-REQ-025', 'SFMC-REQ-026', 'SFMC-REQ-027', 'SFMC-REQ-028', 'SFMC-REQ-029', 'SFMC-REQ-030', 'SFMC-REQ-031', 'SFMC-REQ-032', 'SFMC-REQ-033', 'SFMC-REQ-034', 'SFMC-REQ-035', 'SFMC-REQ-036', 'SFMC-REQ-037', 'SFMC-REQ-038', 'SFMC-REQ-039', 'SFMC-REQ-040', 'SFMC-REQ-041', 'SFMC-REQ-042', 'SFMC-REQ-043', 'SFMC-REQ-044', 'SFMC-REQ-045', 'SFMC-REQ-046', 'SFMC-REQ-047', 'SFMC-REQ-048', 'SFMC-REQ-049', 'SFMC-REQ-050', 'SFMC-REQ-051', 'SFMC-REQ-052', 'SFMC-REQ-053', 'SFMC-REQ-054']
['Position Display', 'Navigation Update', 'Destination Selection', 'Route Calculation', 'Autopilot', 'Autopilot Arrival', 'Engine Statu

In [38]:
#extending the section list by making a list called extended_section_list where it would add the sections based on the number of SRSs that it has
#section_list would be used to added the content of that section with an index that is seperate that will be increased

AmountofTimes = [6, 4, 6, 5, 4, 5, 5, 4, 4, 4, 3, 4]
extended_section_list = []
index = 0

for i in section_list:
    for r in range(0, AmountofTimes[index]):
      extended_section_list.append(i)
    index = index + 1

print(extended_section_list)
print(len(extended_section_list))

['3.1 Navigation ', '3.1 Navigation ', '3.1 Navigation ', '3.1 Navigation ', '3.1 Navigation ', '3.1 Navigation ', '3.2 Propulsion ', '3.2 Propulsion ', '3.2 Propulsion ', '3.2 Propulsion ', '3.3 Targeting and Weapons ', '3.3 Targeting and Weapons ', '3.3 Targeting and Weapons ', '3.3 Targeting and Weapons ', '3.3 Targeting and Weapons ', '3.3 Targeting and Weapons ', '3.4 Defensive Systems ', '3.4 Defensive Systems ', '3.4 Defensive Systems ', '3.4 Defensive Systems ', '3.4 Defensive Systems ', '3.5 Communications ', '3.5 Communications ', '3.5 Communications ', '3.5 Communications ', '3.6 Vehicle Health Monitoring ', '3.6 Vehicle Health Monitoring ', '3.6 Vehicle Health Monitoring ', '3.6 Vehicle Health Monitoring ', '3.6 Vehicle Health Monitoring ', '4. Emergency Operation ', '4. Emergency Operation ', '4. Emergency Operation ', '4. Emergency Operation ', '4. Emergency Operation ', '5. Performance Requirements ', '5. Performance Requirements ', '5. Performance Requirements ', '5. Pe

In [39]:
#transforming the lists made in set_text into a dataframe
np_revised_req_id_list = np.array(revised_req_id_list)
np_revised_title_list = np.array(revised_title_list)
np_text_list = np.array(text_list)
np_section_list = np.array(extended_section_list)

data = {
    'req_id': np_revised_req_id_list,
    'title': np_revised_title_list,
    'text': np_text_list,
    'section': np_section_list
}

df = pd.DataFrame({key: pd.Series(value) for key, value in data.items()})
print(df)

          req_id                      title  \
0   SFMC-REQ-001           Position Display   
1   SFMC-REQ-002          Navigation Update   
2   SFMC-REQ-003      Destination Selection   
3   SFMC-REQ-004          Route Calculation   
4   SFMC-REQ-005                  Autopilot   
5   SFMC-REQ-006          Autopilot Arrival   
6   SFMC-REQ-007              Engine Status   
7   SFMC-REQ-008               Fuel Warning   
8   SFMC-REQ-009      Critical Fuel Warning   
9   SFMC-REQ-010             Engine Failure   
10  SFMC-REQ-011           Target Detection   
11  SFMC-REQ-012           Target Selection   
12  SFMC-REQ-013         Target Information   
13  SFMC-REQ-014      Weapons Authorization   
14  SFMC-REQ-015        Friendly Spacecraft   
15  SFMC-REQ-016      Targeting Performance   
16  SFMC-REQ-017             Threat Warning   
17  SFMC-REQ-018            Threat Priority   
18  SFMC-REQ-019          Shield Activation   
19  SFMC-REQ-020  Automatic Countermeasures   
20  SFMC-REQ-

## Converting the newly made dataframe into a csv file

Now that the dataframe has been made with all of the element of the text in the right place, the csv file can be made

The csv will have the same columns as the dataframe which are

req_id

title

text

section

In [40]:
CSVCols = ["req_id", "title", "text", "section"]

df[CSVCols].to_csv("../data/srs_requirements.csv", index=False)

## Bring in addressable_clauses.csv

This part will focus on importing the csv file in and then converting it to a dataframe

In [41]:
dfcsv = pd.read_csv("../data/addressable_clauses.csv")
print(dfcsv.shape)
print(dfcsv)

(62, 10)
                  chunk_id   swe_id section  \
0    npr-3.7.3-requirement  SWE-134   3.7.3   
1   npr-3.10.1-requirement  SWE-147  3.10.1   
2   npr-3.11.8-requirement  SWE-210  3.11.8   
3          npr-3.11.8-note  SWE-210  3.11.8   
4           npr-4.1.2-note  SWE-050   4.1.2   
..                     ...      ...     ...   
57      swehb-3.8-guidance      NaN     3.8   
58      swehb-6.1-guidance      NaN     6.1   
59      swehb-7.1-guidance      NaN     7.1   
60  npr-3.12.1-requirement  SWE-052  3.12.1   
61         npr-3.12.1-note  SWE-052  3.12.1   

                                                 text       source  \
0   If a project has safety-critical software or m...  NPR 7150.2D   
1   The project manager shall specify reusability ...  NPR 7150.2D   
2   The project manager shall identify software re...  NPR 7150.2D   
3   Monitoring of key software observables (e.g., ...  NPR 7150.2D   
4   The software technical requirements definition...  NPR 7150.2D   
..    

2. Final dataframe → data/requirement_clause_pairs.csv
Every requirement paired with every clause: 54 × 62 = 3,348 rows, columns req_id, chunk_id.
This is what the baseline scores: for each pair, does the clause apply, and is it Meets / Partial / Gap.
Keep only the two IDs; the text can be joined from the two files when needed.

In [42]:
Finaldf = pd.merge(df, dfcsv, how='cross')

In [43]:
print(Finaldf.shape)

(3348, 14)


In [44]:
Finaldf.to_csv("../data/requirement_clause_pairs.csv", index=False)